In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# Cargar el subset que guardaste
df = pd.read_csv(r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-reducido-merge\df_subset_pa_ap.csv")
print(f"Total imágenes: {len(df)}")

# Las 7 clases con suficientes positivos
TARGET_CLASSES = ['No Finding', 'Cardiomegaly', 'Pleural Effusion', 
                  'Atelectasis', 'Consolidation', 'Pneumonia', 'Support Devices']

# U-zeros: -1.0 (incierto) y NaN → 0; solo 1.0 cuenta como positivo
for c in TARGET_CLASSES:
    df[c] = df[c].fillna(0).replace(-1.0, 0).astype(int)

# Verificar
print("\nDistribución final (positivos por clase):")
print(df[TARGET_CLASSES].sum())

Total imágenes: 315

Distribución final (positivos por clase):
No Finding          215
Cardiomegaly         25
Pleural Effusion     27
Atelectasis          31
Consolidation         4
Pneumonia             9
Support Devices      48
dtype: int64


In [3]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, groups=df['subject_id']))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(df_train)}  ({df_train['subject_id'].nunique()} pacientes)")
print(f"Val:   {len(df_val)}  ({df_val['subject_id'].nunique()} pacientes)")

# Verificar que no hay solapamiento de pacientes
overlap = set(df_train['subject_id']) & set(df_val['subject_id'])
print(f"Pacientes solapados (debe ser 0): {len(overlap)}")

Train: 252  (226 pacientes)
Val:   63  (57 pacientes)
Pacientes solapados (debe ser 0): 0


In [4]:
# Normalización ImageNet (DenseNet-121 viene preentrenada en ImageNet)
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CXRDataset(Dataset):
    def __init__(self, df, classes, transform):
        self.df = df.reset_index(drop=True)
        self.classes = classes
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[self.classes].values.astype(np.float32))
        return img, labels

train_ds = CXRDataset(df_train, TARGET_CLASSES, transform_train)
val_ds   = CXRDataset(df_val,   TARGET_CLASSES, transform_val)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Test rápido: una imagen
x, y = next(iter(train_loader))
print(f"Batch shape: {x.shape}, labels shape: {y.shape}")

Batch shape: torch.Size([16, 3, 224, 224]), labels shape: torch.Size([16, 7])


In [5]:
def build_model(num_classes):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    # Reemplazar la cabeza final para multi-label
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model

model = build_model(num_classes=len(TARGET_CLASSES)).to(device)

# Loss multi-label: BCE con logits (numéricamente estable)
criterion = nn.BCEWithLogitsLoss()

# Optimizador
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print(f"Modelo: DenseNet-121, {sum(p.numel() for p in model.parameters()):,} parámetros")

Modelo: DenseNet-121, 6,961,031 parámetros


In [6]:
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x).cpu().numpy()
            all_logits.append(logits)
            all_labels.append(y.numpy())
    logits = np.vstack(all_logits)
    labels = np.vstack(all_labels)
    
    aucs = {}
    for i, c in enumerate(TARGET_CLASSES):
        if labels[:, i].sum() > 0 and labels[:, i].sum() < len(labels):
            aucs[c] = roc_auc_score(labels[:, i], logits[:, i])
        else:
            aucs[c] = np.nan
    return aucs

NUM_EPOCHS = 10
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
        pbar.set_postfix(loss=loss.item())
    
    epoch_loss /= len(train_ds)
    aucs = evaluate(model, val_loader)
    mean_auc = np.nanmean(list(aucs.values()))
    
    print(f"Epoch {epoch} | Loss: {epoch_loss:.4f} | Mean AUC (val): {mean_auc:.4f}")
    history.append({"epoch": epoch, "loss": epoch_loss, "mean_auc": mean_auc, **aucs})

history_df = pd.DataFrame(history)
print("\nHistorial completo:")
print(history_df)

Epoch 1/10: 100%|██████████| 16/16 [00:34<00:00,  2.17s/it, loss=0.481]


Epoch 1 | Loss: 0.5672 | Mean AUC (val): 0.8199


Epoch 2/10: 100%|██████████| 16/16 [00:29<00:00,  1.82s/it, loss=0.35] 


Epoch 2 | Loss: 0.3627 | Mean AUC (val): 0.8894


Epoch 3/10: 100%|██████████| 16/16 [00:27<00:00,  1.75s/it, loss=0.27] 


...

Epoch 8 | Loss: 0.1163 | Mean AUC (val): 0.8737


Epoch 9/10: 100%|██████████| 16/16 [00:29<00:00,  1.83s/it, loss=0.0642]


Epoch 9 | Loss: 0.1027 | Mean AUC (val): 0.8376


Epoch 10/10: 100%|██████████| 16/16 [00:31<00:00,  1.98s/it, loss=0.0822]


Epoch 10 | Loss: 0.0919 | Mean AUC (val): 0.8334

Historial completo:
   epoch      loss  mean_auc  No Finding  Cardiomegaly  Pleural Effusion  \
0      1  0.567201  0.819929    0.845349      0.813793          0.865497   
1      2  0.362681  0.889389    0.865116      0.820690          0.912281   
2      3  0.266900  0.893563    0.847674      0.882759          0.926901   
3      4  0.219494  0.909816    0.880233      0.924138          0.956140   
4      5  0.180564  0.867605    0.862791      0.900000          0.956140   
5      6  0.152984  0.887509    0.852326      0.910345          0.964912   
6      7  0.129012  0.898231    0.902326      0.917241          0.941520   
7      8  0.116325  0.873682    0.865116      0.834483          0.903509   
8      9  0.102661  0.837634    0.897674      0.755172          0.856725   
9     10  0.091894  0.833414    0.918605      0.641379          0.909357   

   Atelectasis  Consolidation  Pneumonia  Support Devices  
0     0.823045            NaN   0

In [7]:
print("=" * 60)
print("BASELINE — DenseNet-121 sin limpieza")
print("=" * 60)
print(f"Train: {len(df_train)} imgs | Val: {len(df_val)} imgs")
print(f"Mejor mean AUC: {history_df['mean_auc'].max():.4f} (epoch {history_df['mean_auc'].idxmax() + 1})")
print("\nAUC por clase (última epoch):")
for c in TARGET_CLASSES:
    print(f"  {c:<25} {history_df.iloc[-1][c]:.4f}")

# Guardar para comparación futura con el dataset limpio
history_df.to_csv(r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\baseline_history_raw.csv", index=False)
print("\n✓ Historial guardado en baseline_history_raw.csv")

BASELINE — DenseNet-121 sin limpieza
Train: 252 imgs | Val: 63 imgs
Mejor mean AUC: 0.9098 (epoch 4)

AUC por clase (última epoch):
  No Finding                0.9186
  Cardiomegaly              0.6414
  Pleural Effusion          0.9094
  Atelectasis               0.9465
  Consolidation             nan
  Pneumonia                 0.7377
  Support Devices           0.8469

✓ Historial guardado en baseline_history_raw.csv


In [8]:
import torch
print(torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Versión CUDA:", torch.version.cuda)

2.13.0.dev20260513+cu132
CUDA disponible: True
Versión CUDA: 13.2
